# ML.PROCESS_DOCUMENT — BigQuery AI Functions

`ML.PROCESS_DOCUMENT` is a table-valued function that processes documents stored in Cloud Storage using the Document AI API. It sends PDFs, images, and other documents (referenced via a BigQuery object table) to a Document AI processor and returns structured extraction results — entities, key-value pairs, and parsed fields — directly as BigQuery columns.

**When to use it:**
- You need to extract structured data from invoices, receipts, forms, or other documents at scale
- You want to process documents stored in Cloud Storage without leaving BigQuery
- You need OCR, form parsing, or document layout analysis within SQL queries

**Alternatives:**
- [`AI.GENERATE`](../ai_generate/) — Multimodal Gemini prompts for ad-hoc document understanding (no Document AI processor required)

**References:** [Full syntax reference](../../RESOURCES.md) | [Official documentation](https://cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-process-document) | [Setup guide](../../setup/)

---
## Setup

This function requires several resources: a connection, a Document AI processor, a remote model pointing to the processor, documents uploaded to Cloud Storage, and an object table referencing those documents.

> See the [Setup Reference](../../setup/) for details on connections, remote models, object tables, and Document AI processors.

In [1]:
PROJECT_ID = 'statmike-mlops-349915'  # <-- Replace with your project ID
LOCATION = 'US'  # BigQuery dataset location
DATASET_ID = 'bq_ai_functions'  # Shared dataset across all notebooks
CONNECTION_ID = 'bq_ai_functions'  # Shared connection
BUCKET = PROJECT_ID  # GCS bucket (same name as project)

### Environment

> **Already set up the project environment?** The cell below is a no-op — packages are already in your kernel. See the [Setup Reference](../../setup/) for details.
>
> **Running standalone** (Colab, Colab Enterprise, Vertex AI Workbench)? The cell below installs required packages into your current kernel.

In [2]:
import subprocess, sys, shutil

def install(*packages):
    """Install packages using uv (fast) with pip fallback."""
    uv = shutil.which('uv')
    if uv:
        subprocess.check_call([uv, 'pip', 'install', '-q', '--python', sys.executable, *packages])
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])

install(
    'google-cloud-bigquery', 'db-dtypes', 'bigquery-magics', 'tqdm',
    'google-cloud-storage', 'google-cloud-documentai',
)

In [3]:
# Authenticate (Colab only — other environments use Application Default Credentials)
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass  # Not in Colab — ADC is used automatically

In [4]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project=PROJECT_ID)
pd.set_option('display.max_colwidth', None)

# Create the shared dataset (idempotent)
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = LOCATION
client.create_dataset(dataset, exists_ok=True)
print(f'Dataset {PROJECT_ID}.{DATASET_ID} ready')

# Register %%bigquery cell magic (auto-loaded in Colab, needed elsewhere)
%load_ext bigquery_magics

Dataset statmike-mlops-349915.bq_ai_functions ready


In [5]:
import subprocess as _sp, json as _json

# Create connection (idempotent)
_sp.run(['bq', 'mk', '--connection', '--location', LOCATION,
         '--connection_type', 'CLOUD_RESOURCE',
         '--project_id', PROJECT_ID, CONNECTION_ID],
        capture_output=True, text=True)

# Get service account
r = _sp.run(['bq', 'show', '--connection', '--format=json',
             '--project_id', PROJECT_ID, '--location', LOCATION, CONNECTION_ID],
            capture_output=True, text=True, check=True)
sa = _json.loads(r.stdout)['cloudResource']['serviceAccountId']

# Grant required roles to connection service account
for role in ['roles/aiplatform.user', 'roles/storage.objectViewer', 'roles/documentai.apiUser', 'roles/documentai.viewer']:
    _sp.run(['gcloud', 'projects', 'add-iam-policy-binding', PROJECT_ID,
             f'--member=serviceAccount:{sa}', f'--role={role}', '--quiet'],
            capture_output=True, text=True)
print(f'Connection {CONNECTION_ID} ready (SA: {sa})')

Connection bq_ai_functions ready (SA: bqcx-1026793852137-hx0g@gcp-sa-bigquery-condel.iam.gserviceaccount.com)


In [6]:
# give permissions time to propogate
import time
time.sleep(60)

### Step 1 — Upload invoices to GCS

Upload 50 synthetic invoice PDFs to Cloud Storage. The invoices are in the `data/documents/invoices/` directory of this repository.

> **Running from a repo checkout?** The cell below resolves the path automatically.
> **Running in Colab?** Clone the repo first: `!git clone https://github.com/statmike/vertex-ai-mlops.git`

In [ ]:
from google.cloud import storage
from pathlib import Path
from tqdm import tqdm

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET)

# Resolve local path to invoice PDFs
local_dir = Path('../../data/documents/invoices')
if not local_dir.exists():
    local_dir = Path('vertex-ai-mlops/data+ai/bq-ai-functions/data/documents/invoices')
assert local_dir.exists(), f'Invoice directory not found. Clone the repo or adjust the path.'

gcs_prefix = 'bq_ai_functions/ml_process_document/'
pdfs = sorted(local_dir.glob('*.pdf'))

uploaded = 0
for pdf in tqdm(pdfs, desc='Uploading invoices'):
    blob_name = f'{gcs_prefix}{pdf.name}'
    blob = bucket.blob(blob_name)
    if not blob.exists():
        blob.upload_from_filename(str(pdf))
        uploaded += 1

print(f'Uploaded {uploaded} new files ({len(pdfs)} total in gs://{BUCKET}/{gcs_prefix})')

### Step 2 — Create object table

An [object table](https://cloud.google.com/bigquery/docs/object-table-introduction) is an external table that references unstructured data (PDFs, images, etc.) in Cloud Storage. ML.PROCESS_DOCUMENT reads documents through the object table.

In [8]:
# Create object table over the uploaded invoices
client.query(f'''
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoices`
  WITH CONNECTION `{PROJECT_ID}.{LOCATION}.{CONNECTION_ID}`
  OPTIONS (
    object_metadata = 'SIMPLE',
    uris = ['gs://{BUCKET}/bq_ai_functions/ml_process_document/*']
  )
''').result()
print('Object table ml_process_document_invoices ready')

# Verify — show a few rows
client.query(f'''
SELECT uri, content_type, size
FROM `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoices`
ORDER BY uri
LIMIT 5
''').to_dataframe()

Object table ml_process_document_invoices ready


,uri,content_type,size
0,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_001.pdf,application/pdf,17240
1,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_002.pdf,application/pdf,16264
2,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_003.pdf,application/pdf,15007
3,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_004.pdf,application/pdf,21071
4,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_005.pdf,application/pdf,17012


### Step 3 — Create Document AI processor and remote model

ML.PROCESS_DOCUMENT requires a remote model that points to a [Document AI processor](https://cloud.google.com/document-ai/docs/overview). The processor determines what type of extraction to perform (invoice parsing, OCR, form parsing, etc.).

We create an Invoice Parser processor via the `google-cloud-documentai` Python client, then create a BigQuery remote model that references it.

In [9]:
from google.cloud import documentai_v1 as documentai

docai_client = documentai.DocumentProcessorServiceClient()
parent = docai_client.common_location_path(PROJECT_ID, 'us')

# Check if processor already exists (idempotent)
existing = [
    p for p in docai_client.list_processors(parent=parent)
    if p.display_name == 'bq_ai_functions_invoice_parser'
]

if existing:
    processor = existing[0]
    print(f'Processor already exists: {processor.name}')
else:
    processor = docai_client.create_processor(
        parent=parent,
        processor=documentai.Processor(
            display_name='bq_ai_functions_invoice_parser',
            type_='INVOICE_PROCESSOR',
        ),
    )
    print(f'Created processor: {processor.name}')

PROCESSOR_NAME = processor.name

Processor already exists: projects/1026793852137/locations/us/processors/c8aa2e210e22f98d


In [10]:
# Create remote model pointing to the Document AI processor
client.query(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoice_parser`
  REMOTE WITH CONNECTION `{PROJECT_ID}.{LOCATION}.{CONNECTION_ID}`
  OPTIONS (
    REMOTE_SERVICE_TYPE = 'CLOUD_AI_DOCUMENT_V1',
    DOCUMENT_PROCESSOR = '{PROCESSOR_NAME}'
  )
''').result()
print('Model ml_process_document_invoice_parser ready')

Model ml_process_document_invoice_parser ready


---
## Examples — SQL

Progressive examples using ML.PROCESS_DOCUMENT to extract data from invoices.

### 1. Process a single invoice — discover output columns

Process one invoice with `SELECT *` to see all columns the Invoice Parser returns. This is useful for discovering processor-specific fields.

In [11]:
query = f'''
SELECT *
FROM ML.PROCESS_DOCUMENT(
  MODEL `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoice_parser`,
  (SELECT *
   FROM `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoices`
   ORDER BY uri
   LIMIT 1)
)
'''
df = client.query(query).to_dataframe()
print('Columns returned:', list(df.columns))
df

Columns returned: ['ml_process_document_result', 'ml_process_document_status', 'invoice_type', 'currency', 'due_date', 'invoice_date', 'invoice_id', 'net_amount', 'purchase_order', 'receiver_name', 'receiver_tax_id', 'supplier_iban', 'supplier_name', 'supplier_payment_ref', 'supplier_registration', 'supplier_tax_id', 'total_amount', 'total_tax_amount', 'vat', 'payment_terms', 'line_item', 'amount_paid_since_last_invoice', 'carrier', 'currency_exchange_rate', 'delivery_date', 'freight_amount', 'receiver_address', 'receiver_email', 'receiver_phone', 'receiver_website', 'remit_to_address', 'remit_to_name', 'ship_from_address', 'ship_from_name', 'ship_to_address', 'ship_to_name', 'supplier_address', 'supplier_email', 'supplier_phone', 'supplier_website', 'uri', 'generation', 'content_type', 'size', 'md5_hash', 'updated', 'metadata', 'ref']


ml_process_document_result  \
0  {"entities":[{"confidence":0.98181015,"id":"0","mentionText":"767.25","normalizedValue":{"floatValue":767.25,"text":"767.25"},"pageAnchor":{"pageRefs":[{"boundingPoly":{"normalizedVertices":[{"x":0.90216154,"y":0.42637363},{"x":0.94254833,"y":0.42637363},{"x":0.94254833,"y":0.4342857},{"x":0.90216154,"y":0.4342857}]}}]},"textAnchor":{"content":"767.25","textSegments":[{"endIndex":"687","startIndex":"681"}]},"type":"total_tax_amount"},{"confidence":0.96062666,"id":"1","mentionText":"9300.00","normalizedValue":{"floatValue":9300,"text":"9300"},"pageAnchor":{"pageRefs":[{"boundingPoly":{"normalizedVertices":[{"x":0.89533561,"y":0.40351647},{"x":0.9431172,"y":0.40351647},{"x":0.9431172,"y":0.41142857},{"x":0.89533561,"y":0.41142857}]}}]},"textAnchor":{"content":"9300.00","textSegments":[{"endIndex":"679","startIndex":"672"}]},"type":"net_amount"},{"confidence":0.95871139,"id":"2","mentionText":"Net 30","pageAnchor":{"pageRefs":[{"boundingPoly":{"normalizedVertices":[{"x":0.92719001,"y":0.18989012},{"x":0.96302617,"y":0.18989012},{"x":0.96302617,"y":0.19692308},{"x":0.92719001,"y":0.19692308}]}}]},"textAnchor":{"content":"Net 30","textSegments":[{"endIndex":"550","startIndex":"544"}]},"type":"payment_terms"},{"confidence":0.9463706,"id":"3","mentionText":"INV-2024-0013","pageAnchor":{"pageRefs":[{"boundingPoly":{"normalizedVertices":[{"x":0.86405003,"y":0.14241758},{"x":0.96245736,"y":0.14241758},{"x":0.96245736,"y":0.15032966},{"x":0.86405003,"y":0.15032966}]}}]},"textAnchor":{"content":"INV-2024-0013","textSegments":[{"endIndex":"501","startIndex":"488"}]},"type":"invoice_id"},{"confidence":0.93227702,"id":"4","mentionText":"$","normalizedValue":{"text":"USD"},"pageAnchor":{"pageRefs":[{"boundingPoly":{"normalizedVertices":[{"x":0.85096699,"y":0.46241757},{"x":0.86063707,"y":0.46241757},{"x":0.86063707,"y":0.47384617},{"x":0.85096699,"y":0.47384617}]}}]},"textAnchor":{"content":"$","textSegments":[{"endIndex":"699","startIndex":"698"}]},"type":"currency"},{"confidence":0.87449962,"id":"5","mentionText":"2024-07-15","normalizedValue":{"dateValue":{"day":15,"month":7,"year":2024},"text":"2024-07-15"},"pageAnchor":{"pageRefs":[{"boundingPoly":{"normalizedVertices":[{"x":0.90045506,"y":0.15956044},{"x":0.96245736,"y":0.15956044},{"x":0.96245736,"y":0.1665934},{"x":0.90045506,"y":0.1665934}]}}]},"textAnchor":{"content":"2024-07-15","textSegments":[{"endIndex":"520","startIndex":"510"}]},"type":"invoice_date"},{"confidence":0.85911471,"id":"6","mentionText":"2024-08-14","normalizedValue":{"dateValue":{"day":14,"month":8,"year":2024},"text":"2024-08-14"},"pageAnchor":{"pageRefs":[{"boundingPoly":{"normalizedVertices":[{"x":0.90045506,"y":0.17494506},{"x":0.96245736,"y":0.17494506},{"x":0.96245736,"y":0.18197802},{"x":0.90045506,"y":0.18197802}]}}]},"textAnchor":{"content":"2024-08-14","textSegments":[{"endIndex":"536","startIndex":"526"}]},"type":"due_date"},{"confidence":0.85278434,"id":"7","mentionText":"Global Reach Marketing LLC","pageAnchor":{"pageRefs":[{"boundingPoly":{"normalizedVertices":[{"x":0.036405005,"y":0.1410989},{"x":0.20193401,"y":0.1410989},{"x":0.20193401,"y":0.14989011},{"x":0.036405005,"y":0.14989011}]}}]},"textAnchor":{"content":"Global Reach Marketing LLC","textSegments":[{"endIndex":"161","startIndex":"135"}]},"type":"receiver_name"},{"confidence":0.84422427,"id":"8","mentionText":"Nexus Innovations Group","pageAnchor":{"pageRefs":[{"boundingPoly":{"normalizedVertices":[{"x":0.036405005,"y":0.026813187},{"x":0.39078498,"y":0.026813187},{"x":0.39078498,"y":0.049670331},{"x":0.036405005,"y":0.049670331}]}}]},"textAnchor":{"content":"Nexus Innovations Group","textSegments":[{"endIndex":"23"}]},"type":"supplier_name"},{"confidence":0.65467083,"id":"9","normalizedValue":{"text":"invoice_statement"},"pageAnchor":{"pageRefs":[{"confidence":0.85725158}]},"type":"invoice_type"},{"confidence":0.59544498,"id":"10","mentionText":"450 Technology Drive, Suite 200 | Austin, TX 78701","normalize

### 2. Extract specific invoice fields

Select only the processor-specific columns you need. The Invoice Parser returns many fields — `invoice_id`, `supplier_name`, `receiver_name`, `total_amount`, `net_amount`, `total_tax_amount`, `currency`, `invoice_date`, `due_date`, `line_item`, and more. Use Example 1's `SELECT *` output to discover all available columns.

In [12]:
query = f'''
SELECT
  uri,
  invoice_id,
  supplier_name,
  total_amount,
  currency,
  invoice_date,
  due_date
FROM ML.PROCESS_DOCUMENT(
  MODEL `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoice_parser`,
  (SELECT *
   FROM `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoices`
   ORDER BY uri
   LIMIT 5)
)
'''
client.query(query).to_dataframe()

,uri,invoice_id,supplier_name,total_amount,currency,invoice_date,due_date
0,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_001.pdf,INV-2024-0013,Nexus Innovations Group,10067.25,$,2024-07-15,2024-08-14
1,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_002.pdf,INV-2023-0045,Harmony Event Solutions LLC,7605.63,$,2023-10-26,2023-11-25
2,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_003.pdf,INV-2023-0110,Nexus Innovations Group,21217.00,$,2023-10-26,2023-11-25
3,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_004.pdf,INV-2024-0024,Digital Ascent Marketing LLC,10703.25,$,2024-06-15,2024-07-15
4,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_005.pdf,INV-2024-0025,Apex Digital Solutions LLC,17698.88,$,2024-05-20,2024-06-19


### 3. Process all 50 invoices

Process every invoice in the object table. Extract key fields and display as a table.

In [13]:
query = f'''
SELECT
  uri,
  invoice_id,
  supplier_name,
  total_amount,
  currency,
  invoice_date,
  due_date,
  ml_process_document_status
FROM ML.PROCESS_DOCUMENT(
  MODEL `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoice_parser`,
  TABLE `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoices`
)
ORDER BY uri
'''
df_all = client.query(query).to_dataframe()
print(f'{len(df_all)} invoices processed')
df_all

50 invoices processed


,uri,invoice_id,supplier_name,total_amount,currency,invoice_date,due_date,ml_process_document_status
0,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_001.pdf,INV-2024-0013,Nexus Innovations Group,10067.25,$,2024-07-15,2024-08-14,
1,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_002.pdf,INV-2023-0045,Harmony Event Solutions LLC,7605.63,$,2023-10-26,2023-11-25,
2,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_003.pdf,INV-2023-0110,Nexus Innovations Group,21217.00,$,2023-10-26,2023-11-25,
3,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_004.pdf,INV-2024-0024,Digital Ascent Marketing LLC,10703.25,$,2024-06-15,2024-07-15,
4,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_005.pdf,INV-2024-0025,Apex Digital Solutions LLC,17698.88,$,2024-05-20,2024-06-19,
5,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_006.pdf,INV-2024-0032,Innovate Digital Solutions,16291.63,$,2024-04-15,2024-05-15,
6,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_007.pdf,INV-2024-0039,PixelForge Innovations,8600.00,$,2024-04-15,2024-05-15,
7,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_008.pdf,INV-2023-0006,Quantum Innovations Corp.,35690.00,$,2023-11-15,2023-12-15,
8,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_009.pdf,INV-2023-0012,Quantum Solutions LLC,23220.38,$,2023-11-15,2023-12-15,
9,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_010.pdf,INV-2023-0018,InnovateX Solutions LLC,17861.25,$,2023-11-15,2023-12-15,


### 4. Persist results to a table

Use `CREATE TABLE ... AS SELECT` to save extraction output. This avoids re-processing documents and makes results queryable by other users.

In [14]:
query = f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.ml_process_document_results` AS
SELECT
  uri,
  invoice_id,
  supplier_name,
  total_amount,
  currency,
  invoice_date,
  due_date
FROM ML.PROCESS_DOCUMENT(
  MODEL `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoice_parser`,
  TABLE `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoices`
)
'''
client.query(query).result()
print('Results persisted to ml_process_document_results')

# Verify
client.query(f'''
SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.ml_process_document_results`
ORDER BY uri
LIMIT 5
''').to_dataframe()

Results persisted to ml_process_document_results


,uri,invoice_id,supplier_name,total_amount,currency,invoice_date,due_date
0,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_001.pdf,INV-2024-0013,Nexus Innovations Group,10067.25,$,2024-07-15,2024-08-14
1,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_002.pdf,INV-2023-0045,Harmony Event Solutions LLC,7605.63,$,2023-10-26,2023-11-25
2,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_003.pdf,INV-2023-0110,Nexus Innovations Group,21217.00,$,2023-10-26,2023-11-25
3,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_004.pdf,INV-2024-0024,Digital Ascent Marketing LLC,10703.25,$,2024-06-15,2024-07-15
4,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_005.pdf,INV-2024-0025,Apex Digital Solutions LLC,17698.88,$,2024-05-20,2024-06-19


### 5. Using PROCESS_OPTIONS — page selection

Use `PROCESS_OPTIONS` to control which pages to process. The `fromStart` option processes only the first N pages of each document.

In [15]:
query = f'''
SELECT
  uri,
  invoice_id,
  supplier_name,
  total_amount
FROM ML.PROCESS_DOCUMENT(
  MODEL `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoice_parser`,
  (SELECT *
   FROM `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoices`
   ORDER BY uri
   LIMIT 3),
  PROCESS_OPTIONS => JSON '{{"fromStart": 1}}'
)
'''
client.query(query).to_dataframe()

,uri,invoice_id,supplier_name,total_amount
0,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_001.pdf,INV-2024-0013,Nexus Innovations Group,10067.25
1,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_002.pdf,INV-2023-0045,Harmony Event Solutions LLC,7605.63
2,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_003.pdf,INV-2023-0110,Nexus Innovations Group,21217.00


---
## Examples — `%%bigquery` Magics

`%%bigquery` magics cannot interpolate Python variables in the SQL body. Since ML.PROCESS_DOCUMENT requires `MODEL` and `TABLE` references with project/dataset, most examples are better run via `client.query()` with f-strings. Here we show one example with hardcoded references.

### Process invoices with `%%bigquery`

In [16]:
%%bigquery --project {PROJECT_ID}

SELECT
  uri,
  invoice_id,
  supplier_name,
  total_amount
FROM ML.PROCESS_DOCUMENT(
  MODEL `statmike-mlops-349915.bq_ai_functions.ml_process_document_invoice_parser`,
  (SELECT *
   FROM `statmike-mlops-349915.bq_ai_functions.ml_process_document_invoices`
   ORDER BY uri
   LIMIT 3)
)

Query is running:   0%|          |

Downloading:   0%|          |

,uri,invoice_id,supplier_name,total_amount
0,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_001.pdf,INV-2024-0013,Nexus Innovations Group,10067.25
1,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_002.pdf,INV-2023-0045,Harmony Event Solutions LLC,7605.63
2,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_003.pdf,INV-2023-0110,Nexus Innovations Group,21217.00


---
## Examples — BigFrames

ML.PROCESS_DOCUMENT has no BigFrames equivalent. Use `session.read_gbq_query()` to execute the SQL from BigFrames.

In [17]:
import bigframes.pandas as bpd

bpd.options.bigquery.project = PROJECT_ID
bpd.options.bigquery.location = LOCATION

session = bpd.get_global_session()
df_bf = session.read_gbq_query(f'''
SELECT
  uri,
  invoice_id,
  supplier_name,
  total_amount,
  currency
FROM ML.PROCESS_DOCUMENT(
  MODEL `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoice_parser`,
  (SELECT *
   FROM `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoices`
   ORDER BY uri
   LIMIT 5)
)
''')
df_bf.to_pandas()

,uri,invoice_id,supplier_name,total_amount,currency
0,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_001.pdf,INV-2024-0013,Nexus Innovations Group,10067.25,$
1,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_002.pdf,INV-2023-0045,Harmony Event Solutions LLC,7605.63,$
2,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_003.pdf,INV-2023-0110,Nexus Innovations Group,21217.00,$
3,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_004.pdf,INV-2024-0024,Digital Ascent Marketing LLC,10703.25,$
4,gs://statmike-mlops-349915/bq_ai_functions/ml_process_document/invoice_005.pdf,INV-2024-0025,Apex Digital Solutions LLC,17698.88,$


---
## Cleanup

Drop resources created by this notebook. Shared resources (dataset, connection) are left for other notebooks.

In [ ]:
# Drop notebook-specific resources
for resource in [
    f'DROP TABLE IF EXISTS `{PROJECT_ID}.{DATASET_ID}.ml_process_document_results`',
    f'DROP EXTERNAL TABLE IF EXISTS `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoices`',
    f'DROP MODEL IF EXISTS `{PROJECT_ID}.{DATASET_ID}.ml_process_document_invoice_parser`',
]:
    client.query(resource).result()
print('Notebook resources cleaned up')

# Optional: delete GCS files
# storage_client = storage.Client(project=PROJECT_ID)
# bucket = storage_client.bucket(BUCKET)
# blobs = list(bucket.list_blobs(prefix='bq_ai_functions/ml_process_document/'))
# for blob in blobs:
#     blob.delete()
# print(f'Deleted {len(blobs)} files from GCS')

# Optional: delete Document AI processor
# docai_client.delete_processor(name=PROCESSOR_NAME)
# print(f'Deleted processor {PROCESSOR_NAME}')


### Remove all shared resources

When finished with **all** notebooks, uncomment and run the cell below to delete the shared dataset and all tables, models, and other resources within it.

In [ ]:
# client.delete_dataset(
#     f'{PROJECT_ID}.{DATASET_ID}',
#     delete_contents=True,
#     not_found_ok=True
# )
# print(f'Dataset {PROJECT_ID}.{DATASET_ID} deleted')